In [ ]:
# pip install nltk prettytable openpyxl

In [3]:
!pip install torch>=2.6.0

In [3]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
print(torch.__version__)


2.5.1+cu121


In [4]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
import pandas as pd
import torch

# Monkey patch to bypass torch version check
def bypass_torch_load(*args, **kwargs):
    kwargs.pop('weights_only', None)
    return torch._C._load(*args, **kwargs)

torch.load = bypass_torch_load

from UniEval.metric.evaluator import get_evaluator
from UniEval.utils import convert_to_json

df = pd.read_excel("eval_results.xlsx")
evaluator = get_evaluator('dialogue')

all_scores = []
for _, row in df.iterrows():
    data = convert_to_json([row['responses']], [row['queries']], [row['queries']])
    scores = evaluator.evaluate(data, print_result=False)[0]
    all_scores.append({k: round(float(v), 3) for k, v in scores.items()})

for key in all_scores[0].keys():
    df[key] = [score[key] for score in all_scores]

df.to_excel("eval_results.xlsx", index=False)
